In [21]:
from pyspark.sql import SparkSession

In [22]:
spark = SparkSession.builder \
    .master("local") \
    .appName("Local Spark") \
    .config('spark.ui.port', '4040') \
    .getOrCreate()


In [23]:
sc = spark.sparkContext

In [24]:
class FlightsParser:
    @staticmethod
    def _clean_str(value, to_lower=True):
        if value is None or str(value).strip() == "":
            return "unknown"
        cleaned = str(value).strip()
        return cleaned.lower() if to_lower else cleaned

    @staticmethod
    def _clean_float(value):
        try:
            return float(value) if value is not None else 0.0
        except (ValueError, TypeError):
            return 0.0

    @staticmethod
    def parse_airports(row):
        """
        Output: (id, name, city, state) - Tupla piatta
        """
        try:
            return (
                FlightsParser._clean_str(row[0]), # ID
                FlightsParser._clean_str(row[5]), # Name
                FlightsParser._clean_str(row[3]), # City
                FlightsParser._clean_str(row[4])  # State
            )
        except Exception:
            return None

    @staticmethod
    def parse_flights(row):
        """
        Output: (origin_id, airline, year, month, arr_delay, air_delay, weath_delay, nas_delay, sec_delay, late_delay)
        """
        try:
            origin_id = FlightsParser._clean_str(row[8])
            airline = FlightsParser._clean_str(row[5])
            year = int(row[0])
            month = int(row[1])

            arr_delay = max(0.0, FlightsParser._clean_float(row[24]))
            air_delay = FlightsParser._clean_float(row[29])
            weath_delay = FlightsParser._clean_float(row[30])
            nas_delay = FlightsParser._clean_float(row[31])
            sec_delay = FlightsParser._clean_float(row[32])
            late_delay = FlightsParser._clean_float(row[33])

            return (
                origin_id, airline, year, month, 
                arr_delay, air_delay, weath_delay, 
                nas_delay, sec_delay, late_delay
            )
        except Exception:
            return None
            

In [27]:
airports_df = (
    spark.read
    .option("header", "true")
    .csv("../../../../datasets/airports.csv")
)

flights_df = (
    spark.read
    .option("header", "true")
    .csv("../../../../datasets/flights_sample.csv")
)

airports = (
    airports_df
    .rdd
    .map(FlightsParser.parse_airports)
    .filter(lambda x: x is not None)
)

flights = (
    flights_df
    .rdd
    .map(FlightsParser.parse_flights)
    .filter(lambda x: x is not None)
)

rddA = airports.\
    map(lambda x: (x[0], (x[1], x[2], x[3]))).\
    reduceByKey(lambda x, _: x)

rddF = flights.\
    map(lambda x: (x[0], (x[1], x[2], x[3], x[4], x[5], x[6], x[7], x[8], x[9])))

print("Prime righe flights parsate:")
print(flights.take(3))

print("Numero voli validi:", rddF.count())
print("Numero aeroporti unici:", rddA.count())

Prime righe flights parsate:
[('10721', 'ea', 1988, 1, 17.0, 0.0, 0.0, 0.0, 0.0, 0.0), ('10721', 'ea', 1988, 1, 63.0, 0.0, 0.0, 0.0, 0.0, 0.0), ('10721', 'ea', 1988, 1, 6.0, 0.0, 0.0, 0.0, 0.0, 0.0)]
Numero voli validi: 49999
Numero aeroporti unici: 436


In [39]:
nonOptJob = rddF.\
    filter(lambda x: x[1][3] > 0).\
    join(rddA).\
    map(lambda x: (
        (x[1][0][0], x[1][1][1], x[1][1][2]),
        (
            x[1][0][3],
            x[1][0][4],
            x[1][0][5],
            x[1][0][6],
            x[1][0][7],
            x[1][0][8],
            1
        )
    )).\
    reduceByKey(lambda x, y: (
        x[0] + y[0],
        x[1] + y[1],
        x[2] + y[2],
        x[3] + y[3],
        x[4] + y[4],
        x[5] + y[5],
        x[6] + y[6]
    )).\
    mapValues(lambda x: (
        x[0] / x[6],
        x[1] / x[6],
        x[2] / x[6],
        x[3] / x[6],
        x[4] / x[6],
        x[5] / x[6]
    )).\
    map(lambda x: x[0] + x[1])

In [40]:
bRddA = sc.broadcast(rddA.collectAsMap())

optJob = rddF.\
    filter(lambda x: x[1][3] > 0).\
    flatMap(lambda x: [
        (
            (x[1][0], airport[1], airport[2]),
            (
                x[1][3],
                x[1][4],
                x[1][5],
                x[1][6],
                x[1][7],
                x[1][8],
                1
            )
        )
    ] if (airport := bRddA.value.get(x[0])) is not None else []).\
    reduceByKey(lambda x, y: (
        x[0] + y[0],
        x[1] + y[1],
        x[2] + y[2],
        x[3] + y[3],
        x[4] + y[4],
        x[5] + y[5],
        x[6] + y[6]
    )).\
    mapValues(lambda x: (
        x[0] / x[6],
        x[1] / x[6],
        x[2] / x[6],
        x[3] / x[6],
        x[4] / x[6],
        x[5] / x[6]
    )).\
    map(lambda x: x[0] + x[1])

In [41]:
non_opt_rows = sorted(
    nonOptJob.collect(),
    key=lambda x: (x[0], x[1], x[2])
)

opt_rows = sorted(
    optJob.collect(),
    key=lambda x: (x[0], x[1], x[2])
)

columns = [
    "Airline",
    "City",
    "State",
    "Avg_Arrival_Delay",
    "Avg_Airline_Delay",
    "Avg_Weather_Delay",
    "Avg_NAS_Delay",
    "Avg_Security_Delay",
    "Avg_Late_Aircraft_Delay"
]


def print_results(title, rows):
    print("\n" + title)
    print("=" * len(title))
    print(" | ".join(columns))
    print("-" * 180)

    for row in rows:
        print(
            f"{row[0]} | {row[1]} | {row[2]} | "
            f"{row[3]:.6f} | {row[4]:.6f} | {row[5]:.6f} | "
            f"{row[6]:.6f} | {row[7]:.6f} | {row[8]:.6f}"
        )

    print(f"\nNumero righe: {len(rows)}")


print_results("Job 1 - regular join", non_opt_rows)
print_results("Job 2 - broadcast join", opt_rows)


Job 1 - regular join
Airline | City | State | Avg_Arrival_Delay | Avg_Airline_Delay | Avg_Weather_Delay | Avg_NAS_Delay | Avg_Security_Delay | Avg_Late_Aircraft_Delay
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
dl | albuquerque, nm | nm | 13.411504 | 0.000000 | 0.000000 | 0.000000 | 0.000000 | 0.000000
dl | amarillo, tx | tx | 14.545455 | 0.000000 | 0.000000 | 0.000000 | 0.000000 | 0.000000
dl | anchorage, ak | ak | 21.175141 | 0.000000 | 0.000000 | 0.000000 | 0.000000 | 0.000000
dl | atlanta, ga | ga | 18.683803 | 0.000000 | 0.000000 | 0.000000 | 0.000000 | 0.000000
dl | augusta, ga | ga | 12.890625 | 0.000000 | 0.000000 | 0.000000 | 0.000000 | 0.000000
dl | austin, tx | tx | 16.753731 | 0.000000 | 0.000000 | 0.000000 | 0.000000 | 0.000000
dl | baltimore, md | md | 16.535714 | 0.000000 | 0.000000 | 0.000000 | 0.000000 | 0.000000
dl | baton rouge, la